# Paper 3 — combined multi-panel figures (7 main + 6 supplementary)

Each figure cell makes ONE multi-panel figure and saves it as PNG (300 dpi) and PDF to `ROOT\output\Paper3_Figures\`.
Run the setup cell first, then any figure cell (each depends only on the setup cell).

* `ROOT` = parent of this notebook's folder (`...\Pranav Paper3_Final`), fallback `C:\Users\sanga\Desktop\Pranav Paper3_Final`.
* Definitions (participants, phases, 26 outcomes, natural-log transforms, phase means, H4 bout-z scaling, FDR families) follow `scripts\statistical_analysis\*.R` and `library\*.R`.
* `merged()` checks that `output\merged_all_11participants_8min.csv` is the analysed file (MD5 starting `d1566f`); if the CSV is regenerated, update that check deliberately.
* Phase colours/order: BikeU #a50f15, WalkU #f08a5d, BikeG #006d2c, WalkG #74c476.

## Setup: imports, paths, study constants, shared helpers, phase colours

In [ ]:
import sys, os, re, hashlib, subprocess, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, Patch, Rectangle
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

from pathlib import Path

# ---- paths ----
# ROOT = project folder "Pranav Paper3_Final". The notebook lives in ROOT\scripts, so ROOT is the parent of the
# notebook's folder; the absolute PC path is the fallback.
_FALLBACK_ROOT = Path(r"C:\Users\sanga\Desktop\Pranav Paper3_Final")
_MERGED_REL = Path("output") / "merged_all_11participants_8min.csv"
ROOT = next((p for p in (Path.cwd().parent, Path.cwd(), _FALLBACK_ROOT) if (p / _MERGED_REL).exists()), _FALLBACK_ROOT)
OUT = ROOT / "output" / "Paper3_Figures"
os.makedirs(OUT, exist_ok=True)
print("ROOT =", ROOT)
print("OUT  =", OUT)

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5, "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5, "axes.linewidth": 0.6,
    "xtick.major.width": 0.5, "ytick.major.width": 0.5, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "savefig.dpi": 300})

# ---- study constants (Paper3_common.R) ----
PARTICIPANTS = ["P4", "P8", "P9", "P10", "P11", "P12", "P13", "P14", "P15", "P16", "P17"]
PHASES = ["BikeU", "WalkU", "BikeG", "WalkG"]
PHASE_COL = {"BikeU": "#a50f15", "WalkU": "#f08a5d", "BikeG": "#006d2c", "WalkG": "#74c476"}
PHASE_LAB = {"BikeU": "Bike-Urban", "WalkU": "Walk-Urban", "BikeG": "Bike-Green", "WalkG": "Walk-Green"}
OUTCOMES = ["empatica_heart_rate", "empatica_eda_tonic", "empatica_eda_phasic", "empatica_skin_temperature",
    "empatica_hrv_td_sdnn", "empatica_hrv_td_rmssd", "empatica_hrv_fd_lf", "empatica_hrv_fd_hf",
    "empatica_hrv_fd_lf_hf_ratio", "empatica_hrv_fd_lfnu", "empatica_hrv_fd_hfnu", "empatica_hrv_fd_vlf",
    "empatica_hrv_fd_total_power",
    "eyetracker_pupil_diameter_avg", "eyetracker_pupil_change_rate", "eyetracker_blink_duration_s",
    "eyetracker_fixation_duration_s", "eyetracker_fixation_rate", "eyetracker_saccade_duration_s",
    "eyetracker_saccade_amplitude", "eyetracker_saccade_peak_velocity", "eyetracker_gaze_velocity",
    "eyetracker_gaze_dispersion", "eyetracker_distance_from_center", "eyetracker_gaze_centrality",
    "eyetracker_eyelid_aperture_avg"]
HRV_FD = OUTCOMES[6:13]
# clean_labels (H1_H3_core.R) without device prefix
LAB = {"empatica_heart_rate": "Heart rate (bpm)", "empatica_eda_tonic": "EDA tonic (SCL)", "empatica_eda_phasic": "EDA phasic (SCR)",
    "empatica_skin_temperature": "Skin temperature (°C)", "empatica_hrv_td_sdnn": "HRV SDNN (ms)", "empatica_hrv_td_rmssd": "HRV RMSSD (ms)",
    "empatica_hrv_fd_lf": "HRV LF power (ms²)", "empatica_hrv_fd_hf": "HRV HF power (ms²)", "empatica_hrv_fd_lf_hf_ratio": "HRV LF/HF ratio",
    "empatica_hrv_fd_lfnu": "HRV LF (nu)", "empatica_hrv_fd_hfnu": "HRV HF (nu)", "empatica_hrv_fd_vlf": "HRV VLF power (ms²)",
    "empatica_hrv_fd_total_power": "HRV total power (ms²)", "eyetracker_pupil_diameter_avg": "Pupil diameter (mm)",
    "eyetracker_pupil_change_rate": "Pupil change rate", "eyetracker_blink_duration_s": "Blink duration (s)",
    "eyetracker_fixation_duration_s": "Fixation duration (s)", "eyetracker_fixation_rate": "Fixation rate (/s)",
    "eyetracker_saccade_duration_s": "Saccade duration (s)", "eyetracker_saccade_amplitude": "Saccade amplitude (°)",
    "eyetracker_saccade_peak_velocity": "Saccade peak velocity (°/s)", "eyetracker_gaze_velocity": "Gaze velocity (°/s)",
    "eyetracker_gaze_dispersion": "Gaze dispersion", "eyetracker_distance_from_center": "Gaze distance from centre (px)",
    "eyetracker_gaze_centrality": "Gaze centrality", "eyetracker_eyelid_aperture_avg": "Eyelid aperture (mm)"}
SURVEYS = ["questionnaire_current_comfort_1_7", "questionnaire_pleasantness_1_5", "questionnaire_stai6_mean_1_5", "questionnaire_prs11_total_mean_1_5"]
SURVEY_LAB = {"questionnaire_current_comfort_1_7": "General comfort (1–7)", "questionnaire_pleasantness_1_5": "Pleasantness (1–5)",
    "questionnaire_stai6_mean_1_5": "State anxiety STAI-6 (1–5)", "questionnaire_prs11_total_mean_1_5": "Restorativeness PRS-11 (1–5)"}

USED = set()   # input files read, for provenance printout

def src(rel, pc_rel):
    """pc_rel = path relative to ROOT, Windows-style (as in the original script); rel is unused (old staging path)."""
    p = ROOT.joinpath(*pc_rel.split("\\"))
    if not p.exists(): raise FileNotFoundError(p)
    USED.add(str(p)); return str(p)

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

# ---------------- data ----------------
_cache = {}
def merged():
    if "m" in _cache: return _cache["m"]
    p = src("merged_all_11participants_8min.csv", r"output\merged_all_11participants_8min.csv")
    assert md5(p).startswith("d1566f"), "merged CSV is not the current d1566f file"
    d = pd.read_csv(p, low_memory=False)
    d = d[d.ParticipantID.isin(PARTICIPANTS) & d.PhaseID.isin(PHASES)].copy()
    d["Datetime"] = pd.to_datetime(d["Datetime"])        # clock time as stored in the CSV (time zone not documented in the scripts)
    d = d.sort_values(["ParticipantID", "PhaseID", "Datetime"])
    d["t_sec"] = d.groupby(["ParticipantID", "PhaseID"]).Datetime.transform(lambda x: (x - x.min()).dt.total_seconds())
    # bilateral / two-sensor averages exactly as Analysis_combined_core.R average_sources (rowMeans, na.rm=TRUE)
    avg = {"atmo_pm2.5_avg": ["atmo_lys_atmotube_left__atmotube_pm2.5", "atmo_lys_atmotube_right__atmotube_pm2.5"],
           "atmo_pm1_avg": ["atmo_lys_atmotube_left__atmotube_pm1", "atmo_lys_atmotube_right__atmotube_pm1"],
           "atmo_pm10_avg": ["atmo_lys_atmotube_left__atmotube_pm10", "atmo_lys_atmotube_right__atmotube_pm10"],
           "lys_lux_avg": ["atmo_lys_LYS1__lys_lux", "atmo_lys_LYS2__lys_lux"],
           "lys_kelvin_avg": ["atmo_lys_LYS1__lys_kelvin", "atmo_lys_LYS2__lys_kelvin"]}
    for k, v in avg.items(): d[k] = d[v].mean(axis=1, skipna=True)
    assert len(d) == 2112 and (d.groupby(["ParticipantID", "PhaseID"]).size() == 48).all()
    _cache["m"] = d; return d

def ln_outcomes():
    p = src("res/Normalization_H1toH4_all_parameters.csv",
            r"output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_all_parameters.csv")
    n = pd.read_csv(p).set_index("column_name")
    tr = n.loc[OUTCOMES, "transformation_in_models"]
    assert set(tr) <= {"Natural log", "No transformation"}
    return [o for o in OUTCOMES if tr[o] == "Natural log"]

def analysis_units(d):
    """Row-level ln(x) (x<=0 -> NA) for 'Natural log' outcomes (paper3_outcome_natural_log), others native."""
    a = d.copy()
    for o in ln_outcomes():
        x = a[o].astype(float); a[o] = np.where(np.isfinite(x) & (x > 0), np.log(x.where(x > 0)), np.nan)
    return a

def phase_means(d, cols):
    return d.groupby(["ParticipantID", "PhaseID"], as_index=False)[cols].mean()

def ulab(o):
    return ("ln " + LAB[o]) if o in ln_outcomes() else LAB[o]

# ---------------- helpers ----------------
def panel_label(ax, s, dx=-0.02, dy=0.01, fig=None, size=10):
    fig = fig or ax.figure
    bb = ax.get_position()
    fig.text(bb.x0 + dx, bb.y1 + dy, s, fontsize=size, fontweight="bold", ha="right", va="bottom")

def img_axes(ax, path, title=None):
    im = Image.open(path)
    if im.mode == "CMYK": im = im.convert("RGB")
    if im.mode in ("RGBA", "LA", "P"):
        im = im.convert("RGBA"); bg = Image.new("RGBA", im.size, (255, 255, 255, 255)); im = Image.alpha_composite(bg, im).convert("RGB")
    ax.imshow(np.asarray(im)); ax.set_axis_off()
    if title: ax.set_title(title, loc="left", fontsize=8)
    return im.size

def save(fig, name):
    png = OUT / (name + ".png"); pdf = OUT / (name + ".pdf")
    fig.savefig(png, dpi=300, facecolor="white"); fig.savefig(pdf, facecolor="white")
    plt.close(fig); print("saved", png, "and", pdf)
    try:   # show the saved PNG inline when run in Jupyter
        from IPython.display import display, Image as _IPImage
        display(_IPImage(filename=str(png), width=850))
    except Exception:
        pass

def bh(p):
    p = np.asarray(p, float); out = np.full(p.shape, np.nan); ok = np.isfinite(p); pp = p[ok]; n = len(pp)
    if n == 0: return out
    o = np.argsort(pp); q = pp[o] * n / np.arange(1, n + 1); q = np.minimum.accumulate(q[::-1])[::-1]
    r = np.empty(n); r[o] = np.minimum(q, 1); out[ok] = r; return out

def phase_legend(ax, loc="best", ncol=1, **kw):
    h = [Line2D([], [], marker="o", ls="", color=PHASE_COL[p], label=PHASE_LAB[p] + f" ({p})", markersize=4.5) for p in PHASES]
    return ax.legend(handles=h, loc=loc, ncol=ncol, frameon=False, **kw)

def strip_box(ax, data_by_phase, pos=None, width=0.6, log=False):
    pos = pos or list(range(len(PHASES)))
    vals = [np.asarray(v, float)[np.isfinite(np.asarray(v, float))] for v in data_by_phase]
    bp = ax.boxplot(vals, positions=pos, widths=width, patch_artist=True, showfliers=False, whis=(5, 95),
                    medianprops=dict(color="black", lw=0.8), boxprops=dict(lw=0.5), whiskerprops=dict(lw=0.5), capprops=dict(lw=0.5))
    for b, p in zip(bp["boxes"], PHASES): b.set_facecolor(PHASE_COL[p]); b.set_alpha(0.45)
    return bp

## Figure 1 — Study design (route map, session timeline, sensor setup)

Inputs (relative to `ROOT`):
* `Figures\route_maps\Route Plan for Experiment_standard_09Jun2026.jpg`
* `Figures\manuscript_figures\Experiment timeline.png`
* `Figures\sensor_setup\Experiment Sensor Setup.jpg`

In [ ]:
# Figure 1 — study design (existing artwork)
def fig1():
    route = src("art/route_map.jpg", r"Figures\route_maps\Route Plan for Experiment_standard_09Jun2026.jpg")
    tl = src("art/Experiment timeline.png", r"Figures\manuscript_figures\Experiment timeline.png")
    sens = src("art/Experiment Sensor Setup.jpg", r"Figures\sensor_setup\Experiment Sensor Setup.jpg")
    fig = plt.figure(figsize=(7.2, 5.5))
    ax_a = fig.add_axes([0.035, 0.455, 0.575, 0.535]); img_axes(ax_a, route)
    ax_b = fig.add_axes([0.635, 0.52, 0.36, 0.40]); img_axes(ax_b, tl)
    ax_c = fig.add_axes([0.035, 0.01, 0.96, 0.42]); img_axes(ax_c, sens)
    for ax, s_ in [(ax_a, "a"), (ax_b, "b"), (ax_c, "c")]:
        bb = ax.get_window_extent().transformed(fig.transFigure.inverted())
        fig.text(bb.x0 - 0.005, bb.y1, s_, fontsize=10, fontweight="bold", ha="right", va="top")
    save(fig, "Figure1_study_design")

fig1()

## Figure 2 — Analysis flow (schematic)

Inputs (relative to `ROOT`):
* `none (schematic; facts taken from the R scripts)`

In [ ]:
# Figure 2 — analysis flow (schematic; every number checked against data/outputs)
def fig2():
    d = merged()
    n_rows = len(d); n_cells = d.groupby(["ParticipantID", "PhaseID"]).ngroups
    lnout = ln_outcomes()
    h1 = pd.read_csv(src("res/H1_Contrasts.csv", r"output\statistical_analysis\results\H1_H3_phaseavg\H1_Contrasts.csv"))
    h2 = pd.read_csv(src("res/H2_Physio_Psycho_Concordance.csv", r"output\statistical_analysis\results\H1_H3_phaseavg\H2_Physio_Psycho_Concordance.csv"))
    snd = pd.read_csv(src("res/H4snd/base/H4_sound_exposure_tests.csv", r"output\statistical_analysis\results\archive\superseded_sound_20261101\H4_timeseries_sound\H4_sound_exposure_tests.csv"))
    n_sound = snd.sound_variable.nunique(); n_sound_tests = n_sound * snd.outcome.nunique()
    assert n_rows == 2112 and n_cells == 44 and len(lnout) == 8 and set(h1.Family_Size) == {26} and len(h2) == 104 and n_sound_tests == 572
    fig = plt.figure(figsize=(7.2, 5.6)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_axis_off()

    def box(x, y, w, h, title, body, fc, ec="#333333", tsize=7.2, bsize=6.0):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.3,rounding_size=1.2", fc=fc, ec=ec, lw=0.7))
        ax.text(x + 1.0, y + h - 1.2, title, fontsize=tsize, fontweight="bold", va="top")
        ax.text(x + 1.0, y + h - 4.6, body, fontsize=bsize, va="top", linespacing=1.25)

    def arrow(x0, y0, x1, y1):
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", lw=0.8, color="#333333", shrinkA=0, shrinkB=0))

    # row 1: data
    box(1, 84, 31, 14, "10 s epochs (merged CSV)",
        f"11 participants × 4 core phases\n(BikeU, WalkU, BikeG, WalkG)\nmatched 8 min: 48 epochs per bout\n= {n_rows} rows (MD5 d1566f…)", "#f0f0f0")
    box(35, 84, 30, 14, "Outcomes & transforms",
        "26 physiological outcomes:\n13 Empatica + 13 eye-tracker\nnatural log for 8 (EDA tonic, SDNN,\nRMSSD, LF, HF, LF/HF, VLF, total)", "#f0f0f0")
    box(68, 84, 31, 14, f"{n_cells} participant × phase cells",
        "phase means (ln applied per epoch,\nthen averaged); 4 surveys per bout\n(comfort, pleasantness, STAI-6,\nPRS-11)", "#f0f0f0")
    arrow(32.3, 91, 34.7, 91); arrow(65.3, 91, 67.7, 91)
    # bus to confirmatory row
    ax.plot([12.5, 86], [81.3, 81.3], color="#333333", lw=0.8); ax.plot([50, 50], [83.7, 81.3], color="#333333", lw=0.8)
    for x in (12.5, 37, 61, 86): arrow(x, 81.3, x, 79.4)
    ax.text(1, 81.9, "Confirmatory hypotheses", fontsize=7.5, fontweight="bold", color="#08519c")
    box(1, 58, 23, 21, "H1  green − urban",
        "BikeG − BikeU, WalkG − WalkU\nlme(y ~ PhaseID,\n  random = ~1 | participant)\non 44 phase means\nBH-FDR per contrast (26 outcomes)\nsensitivity: within-person z", "#deebf7", bsize=5.8)
    box(25.5, 58, 23, 21, "H3  walk − bike",
        "WalkU − BikeU, WalkG − BikeG\nsame model as H1\nBH-FDR per contrast (26 outcomes)\neffect size: paired SMD\n(Bonett 2008)", "#deebf7", bsize=5.8)
    box(50, 58, 22, 21, "H2  survey ↔ physiology",
        "rmcorr on 44 phase means\n4 surveys × 26 outcomes\n= 104 tests\none BH-FDR family", "#deebf7", bsize=5.8)
    box(73.5, 58, 25.5, 21, "H4  heat × noise",
        "Primary: 10 s time series, Eq1\nY ~ UTCI × dBA + solar Gh + SVF\n + PM2.5 + lux + colour T + GVI\n + phase (sum-to-zero)\nrandom ~1 | participant/phase,\nAR(1) within bout, CR2 inference\nBH per term × equation (26 outc.)", "#deebf7", bsize=5.5)
    # sensitivity
    box(50, 40, 49, 15, "H4 sensitivity analyses (separate FDR families)",
        "• Eq2–7: + sound mains; air T/RH/wind/MRT; HUMIDEX; PET; PM1; PM10\n"
        "• Phase-mean H4 (44 participant × phase cells) = sensitivity, not primary\n"
        "• Lag-60 s exposure sensitivity (BH per term/equation)\n"
        f"• Sound substitution: {n_sound} acoustic measures each replace dBA in Eq1\n"
        f"  ({n_sound} × 26 = {n_sound_tests} tests; BH across {n_sound_tests} per coefficient type);\n"
        "  epochs 10/30/60/300 s, sound lags 30/60 s", "#eef3fa", bsize=5.6)
    arrow(86, 57.7, 86, 55.4)
    # exploratory row fed from the same data (vertical bus in the gap between H3 and H2)
    ax.plot([49.25, 49.25], [81.3, 36.3], color="#333333", lw=0.8)
    ax.plot([16.5, 83.5], [36.3, 36.3], color="#333333", lw=0.8)
    for x in (16.5, 50, 83.5): arrow(x, 36.3, x, 34.4)
    ax.text(1, 37.0, "Exploratory analyses (same epochs/cells; run in parallel, no H5→H6→H9 dependency)", fontsize=7.5, fontweight="bold", color="#a63603")
    box(1, 15, 31, 19, "H5  corridor hotspots",
        "corridor-buffer cells along route\n(walk 11 m, bike 35 m)\nRH1 per phase: median-centred\noutcome ~ cell\nBH-FDR within outcome × phase", "#fee6ce", bsize=5.8)
    box(34.5, 15, 31, 19, "H6  mechanisms (not preregistered)",
        "H6.1 green − urban participant deltas\n  (sign-flip; BH across 8)\nH6.2 lagged first-difference r\n  (0/10/30/60 s)\nH6.3 sound-source slopes (BH across 4)", "#fee6ce", bsize=5.8)
    box(68, 15, 31, 19, "H9  stress composites",
        "physiological, environmental and\neye-tracker composites per\nparticipant × phase\npairwise rmcorr; one BH-FDR family", "#fee6ce", bsize=5.8)
    ax.set_ylim(13.5, 99.5)
    save(fig, "Figure2_analysis_flow")

fig2()

## Figure 3 — Exposures by phase and the UTCI × dBA plane

Inputs (relative to `ROOT`):
* `output\merged_all_11participants_8min.csv`

In [ ]:
# Figure 3 — exposures by phase (10 s epochs) + UTCI × dBA plane (participant phase means)
EXPOSURES = [("ucm_UTCI", "UTCI (°C)", False), ("ucm_AIR_T_mid", "Air temperature (°C)", False),
             ("ucm_MRT", "MRT (°C)", False), ("ucm_SND_dBA", "Sound level (dBA)", False),
             ("atmo_pm2.5_avg", "PM2.5 (µg/m³)", False), ("lys_lux_avg", "Illuminance (lux)", True),
             ("SVF", "Sky view factor", False), ("gvi_50m", "Green view index (%)", False)]

def fig3():
    d = merged()
    fig = plt.figure(figsize=(7.2, 6.6))
    gs = gridspec.GridSpec(3, 4, figure=fig, left=0.075, right=0.985, top=0.965, bottom=0.085, hspace=0.42, wspace=0.45,
                           height_ratios=[1, 1, 1.35])
    letters = "abcdefghij"
    pm = phase_means(d, [e[0] for e in EXPOSURES])
    rng = np.random.default_rng(1)
    for i, (col, lab, logy) in enumerate(EXPOSURES):
        ax = fig.add_subplot(gs[i // 4, i % 4])
        vals = [d.loc[d.PhaseID == p, col].astype(float).values for p in PHASES]
        if logy: vals = [np.where(v > 0, v, np.nan) for v in vals]
        strip_box(ax, vals)
        for j, p in enumerate(PHASES):
            y = pm.loc[pm.PhaseID == p, col].values
            ax.scatter(j + rng.uniform(-0.17, 0.17, len(y)), y, s=7, color=PHASE_COL[p], edgecolor="black", lw=0.3, zorder=3)
        if logy: ax.set_yscale("log")
        ax.set_xticks(range(4)); ax.set_xticklabels(PHASES, rotation=0)
        ax.set_title(lab, fontsize=7, loc="left")
        n_valid = int(np.isfinite(np.concatenate(vals)).sum())
        ax.text(0.99, 0.99, f"n={n_valid}", transform=ax.transAxes, ha="right", va="top", fontsize=5.5, color="#555555")
        panel_label(ax, letters[i], dx=-0.035)
    # UTCI x dBA plane
    ax = fig.add_subplot(gs[2, 0:2])
    for p in PHASES:
        s = pm[pm.PhaseID == p]
        ax.scatter(s.ucm_UTCI, s.ucm_SND_dBA, s=16, color=PHASE_COL[p], edgecolor="black", lw=0.3, label=PHASE_LAB[p], zorder=3)
        ax.scatter(s.ucm_UTCI.mean(), s.ucm_SND_dBA.mean(), s=90, marker="X", color=PHASE_COL[p], edgecolor="black", lw=0.6, zorder=4)
    ax.set_xlabel("UTCI, participant × phase mean (°C)"); ax.set_ylabel("Sound level, participant × phase mean (dBA)")
    ax.set_title("UTCI × dBA exposure plane (44 cells; × = phase mean)", fontsize=7, loc="left")
    phase_legend(ax, loc="lower right", fontsize=6)
    panel_label(ax, "i", dx=-0.06)
    # within-bout (10 s) UTCI vs dBA, coloured by phase
    ax = fig.add_subplot(gs[2, 2:4])
    for p in PHASES:
        s = d[d.PhaseID == p]
        ax.scatter(s.ucm_UTCI, s.ucm_SND_dBA, s=1.2, color=PHASE_COL[p], alpha=0.35, lw=0, rasterized=True)
    r = d[["ucm_UTCI", "ucm_SND_dBA"]].corr().iloc[0, 1]
    ax.set_xlabel("UTCI, 10 s epoch (°C)"); ax.set_ylabel("Sound level, 10 s epoch (dBA)")
    ax.set_title(f"UTCI × dBA, all {len(d)} epochs (Pearson r = {r:.2f}, descriptive)", fontsize=7, loc="left")
    panel_label(ax, "j", dx=-0.06)
    fig.text(0.5, 0.004, "Boxes: 10 s epochs (median, IQR, whiskers 5–95th pct.); dots: participant × phase means. Air T = ucm_AIR_T_mid; PM2.5 = mean of left/right Atmotube;\n"
             "lux = mean of LYS1/LYS2 (log axis); GVI = gvi_50m. Source: merged_all_11participants_8min.csv (MD5 d1566f…).",
             ha="center", fontsize=5.8, color="#444444")
    save(fig, "Figure3_exposures_by_phase")

fig3()

## Figure 4 — H1 + H3 forest plots (paired SMD, 26 outcomes)

Inputs (relative to `ROOT`):
* `output\statistical_analysis\results\H1_H3_phaseavg\H1_Contrasts.csv`
* `output\statistical_analysis\results\H1_H3_phaseavg\H3_Contrasts.csv`
* `output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_all_parameters.csv`

In [ ]:
# Figure 4 — H1 + H3 forest plot (paired SMD, Bonett 2008; filled = primary-LMM BH q < 0.05)
def fig4():
    h1 = pd.read_csv(src("res/H1_Contrasts.csv", r"output\statistical_analysis\results\H1_H3_phaseavg\H1_Contrasts.csv"))
    h3 = pd.read_csv(src("res/H3_Contrasts.csv", r"output\statistical_analysis\results\H1_H3_phaseavg\H3_Contrasts.csv"))
    t = pd.concat([h1, h3]); t = t[t.Normalization.str.startswith("Primary")]
    assert len(t) == 104 and set(t.Family_Size) == {26}
    panels = [("H1.1", "BikeU_BikeG", "BikeG − BikeU", "Green − urban, bike", PHASE_COL["BikeG"]),
              ("H1.2", "WalkU_WalkG", "WalkG − WalkU", "Green − urban, walk", PHASE_COL["WalkG"]),
              ("H3.1", "BikeU_WalkU", "WalkU − BikeU", "Walk − bike, urban", PHASE_COL["WalkU"]),
              ("H3.2", "BikeG_WalkG", "WalkG − BikeG", "Walk − bike, green", "#4d4d4d")]
    ypos = {o: i + (1.0 if i >= 13 else 0) for i, o in enumerate(OUTCOMES)}
    fig = plt.figure(figsize=(7.2, 5.6))
    gs = gridspec.GridSpec(1, 4, figure=fig, left=0.245, right=0.99, top=0.925, bottom=0.10, wspace=0.10)
    xl = (np.floor(t.SMD_CI95_low.min()) - 0.3, np.ceil(t.SMD_CI95_high.max()) + 3.2)
    for k, (sub, con, lab, ttl, col) in enumerate(panels):
        ax = fig.add_subplot(gs[0, k]); s = t[(t.Sub_Hypothesis == sub) & (t.Contrast == con)].set_index("Outcome").reindex(OUTCOMES)
        assert s.SMD_paired_Bonett.notna().sum() >= 25
        ax.axvline(0, color="#777777", lw=0.6, ls="--")
        ax.axhspan(12.5, 14.5, color="white")
        for o in OUTCOMES:
            r = s.loc[o]; y = ypos[o]
            if not np.isfinite(r.SMD_paired_Bonett): continue
            sig = bool(r.FDR_significant)
            ax.plot([r.SMD_CI95_low, r.SMD_CI95_high], [y, y], color=col, lw=0.9)
            ax.plot(r.SMD_paired_Bonett, y, marker="o", ms=4.2, mfc=col if sig else "white", mec=col, mew=0.9)
            if sig: ax.text(max(r.SMD_CI95_high, 0) + 0.25, y, f"q={r.q_FDR:.3f}" if r.q_FDR >= 0.001 else f"q={r.q_FDR:.0e}", fontsize=5, ha="left", va="center", color="#222222")
        ax.set_ylim(27.7, -0.7); ax.set_xlim(*xl)
        ax.set_yticks([ypos[o] for o in OUTCOMES])
        ax.set_yticklabels([ulab(o) for o in OUTCOMES] if k == 0 else [])
        ax.tick_params(axis="y", length=0 if k else 2)
        ax.set_title(f"{sub}: {lab}\n{ttl}", fontsize=7, color="black")
        ax.set_xlabel("Paired SMD (95% CI)")
        ax.grid(axis="x", color="#eeeeee", lw=0.5); ax.set_axisbelow(True)
        panel_label(ax, "abcd"[k], dx=0.0, dy=0.035)
        if k == 0:
            for (y0, y1, txt) in [(0, 12, "Empatica (13)"), (14, 26, "Eye tracker (13)")]:
                ax.annotate("", xy=(-1.28, y0 - 0.3), xytext=(-1.28, y1 + 0.3), xycoords=ax.get_yaxis_transform(), textcoords=ax.get_yaxis_transform(),
                            arrowprops=dict(arrowstyle="-", lw=0.8, color="#333333"), annotation_clip=False)
                ax.text(-1.33, (y0 + y1) / 2, txt, transform=ax.get_yaxis_transform(), rotation=90, va="center", ha="center", fontsize=7, fontweight="bold")
        ax.axhline(13.0, color="#bbbbbb", lw=0.5)
    leg = [Line2D([], [], marker="o", ls="", mfc="black", mec="black", ms=4, label="BH-FDR q < 0.05 (primary LMM, family = contrast × 26 outcomes)"),
           Line2D([], [], marker="o", ls="", mfc="white", mec="black", ms=4, label="q ≥ 0.05")]
    fig.legend(handles=leg, loc="lower center", ncol=2, frameon=False, fontsize=6.5, bbox_to_anchor=(0.6, 0.0))
    save(fig, "Figure4_H1_H3_forest")

fig4()

## Figure 5 — H2 physiology × psychology rmcorr

Inputs (relative to `ROOT`):
* `output\statistical_analysis\results\H1_H3_phaseavg\H2_Physio_Psycho_Concordance.csv`
* `output\merged_all_11participants_8min.csv`

In [ ]:
# Figure 5 — H2 rmcorr heatmap + two strongest surviving pairs
def rmcorr_fit(df, xcol, ycol):
    """lm(psych ~ factor(ParticipantID) + phys) as in H1_H3_core.R; returns r, slope, centred data."""
    x = df[["ParticipantID", xcol, ycol]].dropna()
    x = x[x.groupby("ParticipantID")[xcol].transform("size") >= 2]
    xc = x[xcol] - x.groupby("ParticipantID")[xcol].transform("mean")
    yc = x[ycol] - x.groupby("ParticipantID")[ycol].transform("mean")
    b = (xc * yc).sum() / (xc ** 2).sum()
    res = yc - b * xc; k = x.ParticipantID.nunique(); dfr = len(x) - k - 1
    se = np.sqrt((res ** 2).sum() / dfr / (xc ** 2).sum()); tv = b / se
    r = np.sign(tv) * np.sqrt(tv ** 2 / (tv ** 2 + dfr))
    return r, b, x.assign(xc=xc, yc=yc), dfr

def fig5():
    h2 = pd.read_csv(src("res/H2_Physio_Psycho_Concordance.csv", r"output\statistical_analysis\results\H1_H3_phaseavg\H2_Physio_Psycho_Concordance.csv"))
    assert len(h2) == 104
    R = h2.pivot(index="Questionnaire", columns="Physio", values="r_rmcorr").reindex(index=SURVEYS, columns=OUTCOMES)
    Q = h2.pivot(index="Questionnaire", columns="Physio", values="q_FDR").reindex(index=SURVEYS, columns=OUTCOMES)
    fig = plt.figure(figsize=(7.2, 6.8))
    ax = fig.add_axes([0.205, 0.53, 0.69, 0.25])
    cmap = plt.get_cmap("RdBu_r")
    im = ax.imshow(R.values, cmap=cmap, vmin=-0.8, vmax=0.8, aspect="auto")
    for i in range(R.shape[0]):
        for j in range(R.shape[1]):
            v = R.values[i, j]; q = Q.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.2f}".replace("0.", ".").replace("-.", "−."), ha="center", va="center", fontsize=4.6,
                        color="white" if abs(v) > 0.5 else "black", fontweight="bold" if q < 0.05 else "normal")
            if np.isfinite(q) and q < 0.05:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=1.3))
    ax.set_yticks(range(4)); ax.set_yticklabels([SURVEY_LAB[s] for s in SURVEYS], fontsize=6.5)
    ax.set_xticks(range(26)); ax.set_xticklabels([ulab(o) for o in OUTCOMES], rotation=55, ha="left", rotation_mode="anchor", fontsize=5.6)
    ax.xaxis.tick_top(); ax.tick_params(axis="x", length=0, pad=1); ax.tick_params(axis="y", length=0)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.axvline(12.5, color="black", lw=0.8)
    cax = fig.add_axes([0.905, 0.53, 0.012, 0.25]); cb = fig.colorbar(im, cax=cax); cb.set_label("rmcorr r", fontsize=6.5); cb.ax.tick_params(labelsize=5.5)
    ax.text(-0.5, 3.75, "Black box / bold = BH-FDR q < 0.05 (one family of 104 tests). Within-person rmcorr across the 4 phases on 44 participant × phase means\n(ln applied per 10 s epoch for log outcomes, then averaged). Vertical line separates Empatica (left) and eye-tracker (right) outcomes.",
            fontsize=5.8, va="top", color="#333333")
    panel_label(ax, "a", dx=-0.12, dy=0.06)
    # strongest surviving pairs (smallest p among q<0.05)
    sig = h2[h2.FDR_significant == True].sort_values("p_value").head(2)
    d = analysis_units(merged()); pm = phase_means(d, OUTCOMES + SURVEYS)
    for k, (_, row) in enumerate(sig.iterrows()):
        ax = fig.add_axes([0.1 + k * 0.49, 0.075, 0.38, 0.32])
        r, b, x, dfr = rmcorr_fit(pm, row.Physio, row.Questionnaire)
        assert abs(r - row.r_rmcorr) < 1e-4 and abs(b - row.Beta) < 1e-4 * max(1, abs(b)), (r, row.r_rmcorr, b, row.Beta)
        x = x.merge(pm[["ParticipantID", "PhaseID", row.Physio, row.Questionnaire]], on=["ParticipantID", row.Physio, row.Questionnaire])
        for p in PHASES:
            s = x[x.PhaseID == p]; ax.scatter(s.xc, s.yc, s=18, color=PHASE_COL[p], edgecolor="black", lw=0.3, zorder=3, label=PHASE_LAB[p])
        xx = np.linspace(x.xc.min(), x.xc.max(), 10); ax.plot(xx, b * xx, color="black", lw=1)
        ax.axhline(0, color="#cccccc", lw=0.5); ax.axvline(0, color="#cccccc", lw=0.5)
        ax.set_xlabel(f"{ulab(row.Physio)}\n(within-person deviation)"); ax.set_ylabel(f"{SURVEY_LAB[row.Questionnaire]}\n(within-person deviation)")
        ax.set_title(f"r_rm = {row.r_rmcorr:.2f} [{row.CI95_low:.2f}, {row.CI95_high:.2f}], q = {row.q_FDR:.1e}, n = {row.N_PhaseBouts} cells / {row.N_Participants} participants",
                     fontsize=6, loc="left")
        if k == 0: phase_legend(ax, loc="upper right", fontsize=5.8)
        panel_label(ax, "bc"[k], dx=-0.06)
    save(fig, "Figure5_H2_rmcorr")

fig5()

## Figure 6 — H4 heat × noise time-series models

Inputs (relative to `ROOT`):
* `output\statistical_analysis\results\Analysis_combined_timeseries\Analysis_combined_timeseries_coefficients.csv, Analysis_combined_timeseries_hypothesis.csv, Analysis_combined_timeseries_lag60_sensitivity.csv`
* `output\statistical_analysis\results\Analysis_combined_phaseavg\Analysis_combined_phaseavg_coefficients.csv`
* `output\statistical_analysis\results\archive\superseded_sound_20261101\H4_timeseries_sound\[base, epoch_30s, epoch_60s, epoch_300s, lag_30s, lag_60s]\H4_sound_exposure_tests.csv`
* `output\merged_all_11participants_8min.csv (shared SD for panel c)`

In [ ]:
# Figure 6 — H4
EQ_SHORT = {"UTCI / atmo_pm2.5_avg": "Eq1 UTCI·PM2.5 (primary)", "UTCI / atmo_pm2.5_avg (sound sensitivity)": "Eq2 + sound mains",
            "AIR_T_mid + AIR_RH + WIND_AWS / atmo_pm2.5_avg": "Eq3 air T/RH/wind/MRT", "HUMIDEX / atmo_pm2.5_avg": "Eq4 HUMIDEX",
            "PET / atmo_pm2.5_avg": "Eq5 PET", "UTCI / atmo_pm1_avg": "Eq6 UTCI·PM1", "UTCI / atmo_pm10_avg": "Eq7 UTCI·PM10"}
TERM_SHORT = {"ucm_UTCI": "UTCI", "ucm_SND_dBA": "dBA", "ucm_UTCI:ucm_SND_dBA": "UTCI × dBA", "ucm_AIR_T_mid": "air T", "ucm_AIR_RH": "RH",
              "ucm_WIND_AWS": "wind", "ucm_MRT": "MRT", "ucm_AIR_T_mid:ucm_SND_dBA": "air T × dBA", "ucm_SND_dBA:ucm_AIR_RH": "RH × dBA",
              "ucm_SND_dBA:ucm_WIND_AWS": "wind × dBA", "ucm_SND_dBA:ucm_MRT": "MRT × dBA", "ucm_HUMIDEX": "HUMIDEX", "ucm_HUMIDEX:ucm_SND_dBA": "HUMIDEX × dBA",
              "ucm_PET": "PET", "ucm_PET:ucm_SND_dBA": "PET × dBA", "vehicle_proportion": "vehicle share", "freq_proportion_20_to_200Hz": "20–200 Hz share",
              "wind_proportion": "wind-noise share", "spectral_centroid_hz": "spectral centroid", "mechanisms_proportion": "mechanisms share",
              "ucm_SUN_Gh": "solar Gh", "SVF": "SVF", "atmo_pm2.5_avg": "PM2.5", "lys_lux_avg": "lux", "lys_kelvin_avg": "colour T", "gvi_50m": "GVI",
              "Phase: WalkG vs mean of phases": "WalkG vs mean", "Phase: WalkU vs mean of phases": "WalkU vs mean",
              "Phase: BikeG vs mean of phases": "BikeG vs mean", "Phase: BikeU vs mean of phases": "BikeU vs mean"}

def bout_shared_sd(d, col):
    x = d[col].astype(float); dev = x - x.groupby([d.ParticipantID, d.PhaseID]).transform("mean")
    return float(np.nanstd(dev[np.isfinite(dev)], ddof=1))

def fig6():
    coef = pd.read_csv(src("res/H4ts/Analysis_combined_timeseries_coefficients.csv", r"output\statistical_analysis\results\Analysis_combined_timeseries\Analysis_combined_timeseries_coefficients.csv"))
    hyp = pd.read_csv(src("res/H4ts/Analysis_combined_timeseries_hypothesis.csv", r"output\statistical_analysis\results\Analysis_combined_timeseries\Analysis_combined_timeseries_hypothesis.csv"))
    lag60 = pd.read_csv(src("res/H4ts/Analysis_combined_timeseries_lag60_sensitivity.csv", r"output\statistical_analysis\results\Analysis_combined_timeseries\Analysis_combined_timeseries_lag60_sensitivity.csv"))
    pavg = pd.read_csv(src("res/Analysis_combined_phaseavg_coefficients.csv", r"output\statistical_analysis\results\Analysis_combined_phaseavg\Analysis_combined_phaseavg_coefficients.csv"))
    eqs = list(EQ_SHORT)
    rows = []
    for e in eqs:
        for t in hyp[hyp.equation == e].term.unique(): rows.append((e, t))
    for t in ["ucm_SUN_Gh", "SVF", "atmo_pm2.5_avg", "lys_lux_avg", "lys_kelvin_avg", "gvi_50m",
              "Phase: WalkG vs mean of phases", "Phase: WalkU vs mean of phases", "Phase: BikeG vs mean of phases", "Phase: BikeU vs mean of phases"]:
        rows.append((eqs[0], t))
    M = np.full((len(rows), 26), np.nan); S = np.zeros((len(rows), 26), bool); NE = np.zeros((len(rows), 26), bool)
    ci = coef.set_index(["equation", "term", "outcome"])
    for i, (e, t) in enumerate(rows):
        for j, o in enumerate(OUTCOMES):
            r = ci.loc[(e, t, o)]
            if r.inference_status != "Tested": NE[i, j] = True; continue
            M[i, j] = np.sign(r.Beta) * -np.log10(max(float(r.q), 1e-12)); S[i, j] = float(r.q) < 0.05
    fig = plt.figure(figsize=(7.2, 10.0))
    ax = fig.add_axes([0.17, 0.445, 0.78, 0.455])
    cmap = plt.get_cmap("PuOr_r"); vmax = 4
    im = ax.imshow(np.clip(M, -vmax, vmax), cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto", interpolation="nearest")
    for i in range(len(rows)):
        for j in range(26):
            if S[i, j]: ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=1.0))
            if NE[i, j]: ax.text(j, i, "×", ha="center", va="center", fontsize=5, color="#888888")
    ylabels, prev = [], None
    for e, t in rows:
        is_cov = (e == eqs[0] and rows.index((e, t)) >= len([r for r in rows if r[1] in hyp[hyp.equation == r[0]].term.unique()]))
        ylabels.append(("Eq1 adj.: " if is_cov else EQ_SHORT[e].split(" ")[0] + ": ") + TERM_SHORT.get(t, t))
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(ylabels, fontsize=5.4)
    ax.set_xticks(range(26)); ax.set_xticklabels([ulab(o) for o in OUTCOMES], rotation=55, ha="left", rotation_mode="anchor", fontsize=5.4)
    ax.xaxis.tick_top(); ax.tick_params(length=0, pad=1)
    for sp in ax.spines.values(): sp.set_visible(False)
    bounds = np.cumsum([len(hyp[hyp.equation == e].term.unique()) for e in eqs])
    for b in bounds: ax.axhline(b - 0.5, color="black", lw=0.6)
    ax.axvline(12.5, color="black", lw=0.6)
    cax = fig.add_axes([0.17, 0.405, 0.30, 0.007]); cb = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_label("sign(β) × −log10(q)   (black box: q < 0.05; ×: not estimable)", fontsize=5.8); cb.ax.tick_params(labelsize=5.5)
    panel_label(ax, "a", dx=-0.12, dy=0.06)
    nsig_hyp = int(((hyp.FDR_significant == "TRUE") | (hyp.FDR_significant == True)).sum())
    fig.text(0.52, 0.412, f"Eq1–Eq7 on 10 s time series (bout-centred exposures, AR(1), CR2).\nq = BH per term within equation across 26 outcomes.\nExposure-term tests with q < 0.05: {nsig_hyp} of {len(hyp)}. Eq3 ln EDA tonic: model not estimable.",
            fontsize=5.6, va="top", color="#333333")

    # ---- b: UTCI × sound across epoch widths / lags ----
    runs = [("base", "10 s"), ("epoch_30s", "30 s"), ("epoch_60s", "60 s"), ("epoch_300s", "300 s"), ("lag_30s", "10 s, lag 30 s"), ("lag_60s", "10 s, lag 60 s")]
    tabs = {}
    for r, _ in runs:
        pc = r"output\statistical_analysis\results\archive\superseded_sound_20261101\H4_timeseries_sound" + ("" if r == "base" else "\\" + r) + r"\H4_sound_exposure_tests.csv"
        tabs[r] = pd.read_csv(src(f"res/H4snd/{r}/H4_sound_exposure_tests.csv", pc))
    sounds = sorted(tabs["base"].sound_variable.unique())
    assert len(sounds) == 22
    B = np.full((22, len(runs)), np.nan); C = np.zeros((22, len(runs)), int); P = np.full((22, len(runs)), np.nan)
    for j, (r, _) in enumerate(runs):
        t = tabs[r][tabs[r].term_type == "thermal_x_sound"]
        for i, sv in enumerate(sounds):
            s = t[t.sound_variable == sv]
            B[i, j] = s.q_FDR.min(); C[i, j] = int((s.q_FDR < 0.05).sum()); P[i, j] = s.p.min()
    # primary dBA interaction: Eq1 10 s, Eq1 lag-60, phase-mean Eq1
    dba = []
    e1 = hyp[(hyp.equation == eqs[0]) & (hyp.term == "ucm_UTCI:ucm_SND_dBA")]; dba.append(("Eq1 10 s (primary)", e1.q.astype(float).min(), int((e1.q.astype(float) < 0.05).sum()), len(e1)))
    l1 = lag60[(lag60.equation_id == 1) & (lag60.term == "ucm_UTCI:ucm_SND_dBA")]; dba.append(("Eq1 exposures lag 60 s", l1.q_lag60.astype(float).min(), int((l1.q_lag60.astype(float) < 0.05).sum()), len(l1)))
    p1 = pavg[(pavg.equation_id == 1) & (pavg.term == "ucm_UTCI:ucm_SND_dBA")]; dba.append(("Eq1 phase means", p1.q.astype(float).min(), int((p1.q.astype(float) < 0.05).sum()), len(p1)))
    axb = fig.add_axes([0.235, 0.075, 0.27, 0.265])
    imb = axb.imshow(-np.log10(P), cmap="Greys", vmin=0, vmax=3, aspect="auto")
    for i in range(22):
        for j in range(len(runs)):
            axb.text(j, i, f"{P[i, j]:.3f}", ha="center", va="center", fontsize=4.4, color="black" if P[i, j] > 0.02 else "white")
    axb.set_yticks(range(22)); axb.set_yticklabels(sounds, fontsize=5.2)
    axb.set_xticks(range(len(runs))); axb.set_xticklabels([l for _, l in runs], rotation=45, ha="right", rotation_mode="anchor", fontsize=5.4)
    axb.tick_params(length=0)
    assert C.sum() == 0
    axb.set_title("UTCI × sound in Eq1 (22 measures each replace dBA).\nCells: min raw p across 26 outcomes. BH q < 0.05: 0 of 572\nin every run; min q per run: " + ", ".join(f"{np.nanmin(B[:, j]):.2f}" for j in range(len(runs))), fontsize=5.6, loc="left")
    panel_label(axb, "b", dx=-0.17, dy=0.045)
    txt = "(b, cont.) UTCI × dBA in the primary family (26 outcomes):\n" + "\n".join(f"  {n}: min q = {q:.2f}; q<0.05 in {k}/{m}" for n, q, k, m in dba)
    fig.text(0.62, 0.345, txt, fontsize=5.4, va="top", ha="left")

    # ---- c: model-implied prediction from exported Eq1 fixed effects ----
    best = e1.assign(pv=e1.p.astype(float)).sort_values("pv").iloc[0]; o = best.outcome
    c1 = coef[(coef.equation == eqs[0]) & (coef.outcome == o)].set_index("term")
    b0, bu, bs, bus = [float(c1.loc[t, "Beta"]) for t in ["Intercept", "ucm_UTCI", "ucm_SND_dBA", "ucm_UTCI:ucm_SND_dBA"]]
    d = merged(); sdU = bout_shared_sd(d, "ucm_UTCI"); sdS = bout_shared_sd(d, "ucm_SND_dBA")
    axc = fig.add_axes([0.68, 0.105, 0.295, 0.15])
    u = np.linspace(-2, 2, 50)
    for sv, ls, lab in [(-1, "--", "dBA −1 SD"), (0, "-", "dBA at bout mean"), (1, ":", "dBA +1 SD")]:
        axc.plot(u * sdU, b0 + bu * u + bs * sv + bus * u * sv, color="black", ls=ls, lw=1.1, label=f"{lab} ({sv * sdS:+.1f} dBA)" if sv else lab)
    axc.set_xlabel("UTCI deviation from bout mean (°C)", fontsize=6.5); axc.set_ylabel(ulab(o), fontsize=6.5); axc.tick_params(labelsize=6)
    axc.legend(frameon=False, fontsize=5.4, loc="best")
    axc.set_title(f"Eq1 fixed-effect prediction, {LAB[o]}\nUTCI×dBA β = {bus:.4f}, p = {float(best.p):.3f}, q = {float(best.q):.2f}", fontsize=6, loc="left")
    panel_label(axc, "c", dx=-0.06, dy=0.028)
    fig.text(0.60, 0.035, f"Exported Eq1 fixed effects (not a refit); covariates at bout mean,\nphase terms 0 (mean of phases). 1 SD bout-centred deviation =\n{sdU:.2f} °C UTCI, {sdS:.2f} dBA. No CI: CR2 covariance not exported.",
             fontsize=5.0, color="#444444")
    print("fig6c outcome", o, "sdU", sdU, "sdS", sdS, "betas", b0, bu, bs, bus)
    save(fig, "Figure6_H4_heat_noise")

fig6()

## Figure 7 — Sound composition by phase

Inputs (relative to `ROOT`):
* `output\merged_all_11participants_8min.csv`

In [ ]:
# Figure 7 — sound composition by phase (10 s epochs, 4 core phases)
YAM = [("vehicle_proportion", "Vehicle", "Vehicle"), ("urban_manmade_positive_proportion", "Urban/man-made (positive)", "Urban/Manmade_positive"),
       ("urban_manmade_neutral_proportion", "Urban/man-made (neutral)", "Urban/Manmade_neutral"),
       ("urban_manmade_negative_proportion", "Urban/man-made (negative)", "Urban/Manmade_negative"),
       ("mechanisms_proportion", "Mechanisms", "Mechanisms"), ("rural_natural_proportion", "Rural/natural", "Rural/Natural"),
       ("wind_proportion", "Wind", "Wind"), ("unidentify_proportion", "Unidentified", "Unidentify")]
YAM_COL = ["#4d4d4d", "#8c6bb1", "#bcbddc", "#e7298a", "#a6761d", "#1b9e77", "#80b1d3", "#d9d9d9"]

def fig7():
    d = merged()
    fig = plt.figure(figsize=(7.2, 8.8))
    gs = fig.add_gridspec(3, 2, left=0.09, right=0.98, top=0.95, bottom=0.075, hspace=0.72, wspace=0.42)
    # a: dominant category per epoch
    ax = fig.add_subplot(gs[0, 0]); cats_present = set(d["category"].dropna())
    assert cats_present <= {c for _, _, c in YAM}, cats_present
    bottom = np.zeros(4)
    for (col, lab, cat), cc in zip(YAM, YAM_COL):
        sh = np.array([(d[d.PhaseID == p]["category"] == cat).sum() / d[d.PhaseID == p]["category"].notna().sum() * 100 for p in PHASES])
        if cat not in cats_present: continue
        ax.bar(range(4), sh, bottom=bottom, color=cc, edgecolor="white", lw=0.3, label=lab, width=0.7)
        for k in range(4):
            if sh[k] >= 6: ax.text(k, bottom[k] + sh[k] / 2, f"{sh[k]:.0f}", ha="center", va="center", fontsize=5.5, color="white" if cc in ("#4d4d4d", "#8c6bb1", "#1b9e77", "#a6761d") else "black")
        bottom += sh
    ax.set_xticks(range(4)); ax.set_xticklabels(PHASES); [t.set_color(PHASE_COL[p]) for t, p in zip(ax.get_xticklabels(), PHASES)]
    ax.set_ylabel("Share of 10 s epochs (%)"); ax.set_ylim(0, 100); ax.set_title("Dominant YAMNet category per epoch", fontsize=7, loc="left")
    ax.legend(frameon=False, fontsize=5.4, bbox_to_anchor=(0.5, -0.12), loc="upper center", ncol=3, handlelength=1, columnspacing=0.8)
    panel_label(ax, "a", dx=-0.05, dy=0.012)
    # b: mean YAMNet frame proportions (multi-label)
    ax = fig.add_subplot(gs[0, 1]); w = 0.2
    rows_sum = d[[c for c, _, _ in YAM]].sum(axis=1, min_count=1)
    for k, p in enumerate(PHASES):
        m = d[d.PhaseID == p][[c for c, _, _ in YAM]].mean()
        ax.barh(np.arange(len(YAM)) + (k - 1.5) * w, m.values, height=w, color=PHASE_COL[p], label=p)
    ax.set_yticks(range(len(YAM))); ax.set_yticklabels([l.replace("Urban/man-made", "Urban") for _, l, _ in YAM], fontsize=5.8); ax.invert_yaxis()
    ax.set_xlabel("Mean proportion of frames in epoch")
    ax.set_title(f"YAMNet frame proportions (multi-label; row sums mean {rows_sum.mean():.2f})", fontsize=6.4, loc="left")
    ax.legend(frameon=False, fontsize=5.4, loc="lower right")
    panel_label(ax, "b", dx=-0.17, dy=0.012)
    # c: three broad frequency bands
    ax = fig.add_subplot(gs[1, 0]); bands = [("freq_proportion_20_to_200Hz", "20–200 Hz", "#08519c"), ("freq_proportion_200_to_2000Hz", "200–2000 Hz", "#6baed6"), ("freq_proportion_2000_to_20000Hz", "2–20 kHz", "#fdae6b")]
    bottom = np.zeros(4)
    for col, lab, cc in bands:
        m = np.array([d[d.PhaseID == p][col].mean() * 100 for p in PHASES])
        ax.bar(range(4), m, bottom=bottom, color=cc, edgecolor="white", lw=0.3, width=0.7, label=lab)
        for k in range(4): ax.text(k, bottom[k] + m[k] / 2, f"{m[k]:.0f}", ha="center", va="center", fontsize=5.5)
        bottom += m
    ax.set_xticks(range(4)); ax.set_xticklabels(PHASES); [t.set_color(PHASE_COL[p]) for t, p in zip(ax.get_xticklabels(), PHASES)]
    ax.set_ylim(0, 100); ax.set_ylabel("Mean energy share (%)"); ax.set_title("Broad frequency-band shares", fontsize=7, loc="left")
    ax.legend(frameon=False, fontsize=5.4, bbox_to_anchor=(0.5, -0.12), loc="upper center", ncol=3, handlelength=1)
    panel_label(ax, "c", dx=-0.05, dy=0.012)
    # d: octave-band profile
    ax = fig.add_subplot(gs[1, 1]); oc = [c for c in d.columns if c.startswith("octave_proportion_")]
    fc = [float(c.replace("octave_proportion_", "").replace("_Hz", "").replace("_", ".")) for c in oc]
    for p in PHASES:
        x = d[d.PhaseID == p][oc]; m = x.mean(); s = x.std(ddof=1) / np.sqrt(x.notna().sum())
        ax.plot(fc, m.values * 100, "-o", color=PHASE_COL[p], ms=2.5, lw=1, label=p)
    ax.set_xscale("log"); ax.set_xticks(fc); ax.set_xticklabels([("%g" % f) if f < 1000 else ("%gk" % (f / 1000)) for f in fc], fontsize=5.6)
    ax.minorticks_off(); ax.set_xlabel("Octave-band centre frequency (Hz)"); ax.set_ylabel("Mean energy share (%)")
    ax.set_title("Octave-band profile", fontsize=7, loc="left"); ax.legend(frameon=False, fontsize=5.4)
    panel_label(ax, "d", dx=-0.06, dy=0.012)
    # e: spectral centroid; f: dBA
    for g, col, lab, ttl, lt in [(gs[2, 0], "spectral_centroid_Hz", "Spectral centroid (Hz)", "Spectral centroid per epoch", "e"),
                                 (gs[2, 1], "ucm_SND_dBA", "Sound level (dBA)", "A-weighted sound level per epoch", "f")]:
        ax = fig.add_subplot(g)
        vals = [d[d.PhaseID == p][col].astype(float).values for p in PHASES]
        vp = ax.violinplot([v[np.isfinite(v)] for v in vals], positions=range(4), widths=0.8, showextrema=False)
        for b, p in zip(vp["bodies"], PHASES): b.set_facecolor(PHASE_COL[p]); b.set_alpha(0.35); b.set_edgecolor("none")
        strip_box(ax, vals, width=0.25)
        for k, v in enumerate(vals): ax.text(k, 1.01, f"med {np.nanmedian(v):.0f}", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=5.4)
        ax.set_xticks(range(4)); ax.set_xticklabels(PHASES); [t.set_color(PHASE_COL[p]) for t, p in zip(ax.get_xticklabels(), PHASES)]
        if col == "spectral_centroid_Hz": ax.set_yscale("log")
        ax.set_ylabel(lab); ax.set_title(ttl, fontsize=7, loc="left", pad=11)
        panel_label(ax, lt, dx=-0.05, dy=0.012)
    n = {p: int(d[d.PhaseID == p]["category"].notna().sum()) for p in PHASES}
    fig.text(0.09, 0.008, "Pooled 10 s epochs, 11 participants, 48 epochs per participant × phase (" + ", ".join(f"{p} n={n[p]}" for p in PHASES) +
             ");\none P13 BikeG epoch has no sound-analysis values. Boxes: median, IQR, 5–95 % whiskers; violins: full distribution. Spectral centroid on log axis.", fontsize=5.6, color="#444444")
    save(fig, "Figure7_sound_composition")

fig7()

## Figure S1 — Participants and data coverage

Inputs (relative to `ROOT`):
* `Figures\manuscript_figures\Participants matched_8min.png`
* `output\merged_all_11participants_8min.csv`

In [ ]:
# Figure S1 — participants and data coverage
def figS1():
    d = merged(); a = analysis_units(d)
    photos = src("art/Participants matched_8min.png", r"Figures\manuscript_figures\Participants matched_8min.png")
    fig = plt.figure(figsize=(7.2, 9.6))
    axa = fig.add_axes([0.06, 0.775, 0.90, 0.215]); img_axes(axa, photos); panel_label(axa, "a", dx=-0.01, dy=-0.01)
    # b: session timeline
    axb = fig.add_axes([0.09, 0.575, 0.88, 0.165])
    order = sorted(PARTICIPANTS, key=lambda p: d[d.ParticipantID == p].Datetime.min().date())
    for i, pid in enumerate(order):
        x = d[d.ParticipantID == pid]; day = x.Datetime.min().normalize()
        for ph in PHASES:
            y = x[x.PhaseID == ph]; t0 = (y.Datetime.min() - day).total_seconds() / 3600; t1 = (y.Datetime.max() - day).total_seconds() / 3600 + 10 / 3600
            axb.barh(i, t1 - t0, left=t0, color=PHASE_COL[ph], height=0.7)
        first = x.sort_values("Datetime").PhaseID.iloc[0]
        axb.text(16.15, i, x.Datetime.min().strftime("%d %b") + ("  green first" if first in ("BikeG", "WalkG") else "  urban first"), va="center", fontsize=5.4)
    axb.set_yticks(range(len(order))); axb.set_yticklabels(order, fontsize=6); axb.invert_yaxis()
    axb.set_xlim(12, 17.2); axb.set_xticks(range(12, 17)); axb.set_xticklabels([f"{h}:00" for h in range(12, 17)])
    axb.set_xlabel("Clock time of 10 s epochs as stored in the merged file (time zone not documented)", fontsize=6.5)
    phase_legend(axb, loc="lower left", ncol=4, fontsize=5.6, bbox_to_anchor=(0.0, 1.0))
    panel_label(axb, "b", dx=-0.05, dy=0.03)
    # c: outcome coverage
    cells = [(p, ph) for p in PARTICIPANTS for ph in PHASES]
    C = np.array([[a[(a.ParticipantID == p) & (a.PhaseID == ph)][o].astype(float).notna().mean() * 100 for p, ph in cells] for o in OUTCOMES])
    axc = fig.add_axes([0.21, 0.255, 0.70, 0.25])
    im = axc.imshow(C, cmap="viridis", vmin=0, vmax=100, aspect="auto", interpolation="nearest")
    axc.set_yticks(range(26)); axc.set_yticklabels([ulab(o) for o in OUTCOMES], fontsize=5.2)
    axc.set_xticks(range(len(cells))); axc.set_xticklabels([ph for _, ph in cells], rotation=90, fontsize=4.2)
    for t, (_, ph) in zip(axc.get_xticklabels(), cells): t.set_color(PHASE_COL[ph])
    for k, p in enumerate(PARTICIPANTS):
        axc.text(k * 4 + 1.5, -0.9, p, ha="center", va="bottom", fontsize=5.4)
        if k: axc.axvline(k * 4 - 0.5, color="white", lw=0.8)
    axc.axhline(12.5, color="white", lw=0.8); axc.tick_params(length=0)
    cb = fig.colorbar(im, cax=fig.add_axes([0.925, 0.255, 0.012, 0.25])); cb.set_label("% of the 48 epochs with a valid value", fontsize=6); cb.ax.tick_params(labelsize=5.5)
    panel_label(axc, "c", dx=-0.17, dy=0.02)
    # d: exposure coverage
    expo = [("ucm_UTCI", "UTCI"), ("ucm_AIR_T_mid", "Air T mid"), ("ucm_MRT", "MRT"), ("ucm_SUN_Gh", "Solar Gh"), ("ucm_SND_dBA", "dBA"),
            ("spectral_centroid_Hz", "Sound spectrum"), ("vehicle_proportion", "YAMNet"), ("atmo_pm2.5_avg", "PM2.5 (L/R mean)"),
            ("lys_lux_avg", "Lux (LYS mean)"), ("lys_kelvin_avg", "Colour T (LYS mean)"), ("SVF", "SVF"), ("gvi_50m", "GVI 50 m")]
    E = np.array([[d[(d.ParticipantID == p) & (d.PhaseID == ph)][c].astype(float).notna().mean() * 100 for p, ph in cells] for c, _ in expo])
    axd = fig.add_axes([0.21, 0.07, 0.70, 0.125])
    axd.imshow(E, cmap="viridis", vmin=0, vmax=100, aspect="auto", interpolation="nearest")
    axd.set_yticks(range(len(expo))); axd.set_yticklabels([l for _, l in expo], fontsize=5.4)
    axd.set_xticks(range(len(cells))); axd.set_xticklabels([ph for _, ph in cells], rotation=90, fontsize=4.2)
    for t, (_, ph) in zip(axd.get_xticklabels(), cells): t.set_color(PHASE_COL[ph])
    for k, p in enumerate(PARTICIPANTS):
        axd.text(k * 4 + 1.5, -0.7, p, ha="center", va="bottom", fontsize=5.4)
        if k: axd.axvline(k * 4 - 0.5, color="white", lw=0.8)
    axd.tick_params(length=0)
    for i in range(len(expo)):
        for j in range(len(cells)):
            if E[i, j] < 99.5: axd.text(j, i, f"{E[i, j]:.0f}", ha="center", va="center", fontsize=3.6, color="white" if E[i, j] < 60 else "black")
    panel_label(axd, "d", dx=-0.17, dy=0.025)
    hrv_av = (d["empatica_hrv_fd_5min_available"].astype(str).str.lower().isin(["true", "1"])).mean() * 100
    fig.text(0.06, 0.006, f"c: ln outcomes counted valid only if > 0 (no EDA tonic value is ≤ 0; gaps are missing values). Eye-tracker: no data for P13 (all phases) or P8 WalkU.\nH4 additionally masks the 7 HRV frequency-domain outcomes where the 5-min window flag is false ({hrv_av:.1f} % of epochs flagged available). d: values printed where coverage < 99.5 %.",
             fontsize=5.3, color="#444444")
    print("S1 coverage min outcome", np.nanmin(C), "min exposure", np.nanmin(E))
    save(fig, "FigureS1_participants_coverage")

figS1()

## Figure S2 — 26 outcomes by phase (participant × phase means)

Inputs (relative to `ROOT`):
* `output\merged_all_11participants_8min.csv`
* `output\statistical_analysis\results\H1_H3_phaseavg\H1_Contrasts.csv, H3_Contrasts.csv`

In [ ]:
# Figure S2 — 26 outcomes by phase (participant × phase means, analysis units)
def figS2():
    d = analysis_units(merged()); pm = phase_means(d, OUTCOMES)
    h1 = pd.read_csv(src("res/H1_Contrasts.csv", r"output\statistical_analysis\results\H1_H3_phaseavg\H1_Contrasts.csv"))
    h3 = pd.read_csv(src("res/H3_Contrasts.csv", r"output\statistical_analysis\results\H1_H3_phaseavg\H3_Contrasts.csv"))
    t = pd.concat([h1, h3]); t = t[t.Normalization.str.startswith("Primary")]
    pos = {p: i for i, p in enumerate(PHASES)}
    fig, axes = plt.subplots(6, 5, figsize=(7.2, 9.6))
    fig.subplots_adjust(left=0.07, right=0.99, top=0.975, bottom=0.04, hspace=0.62, wspace=0.52)
    letters = "abcdefghijklmnopqrstuvwxyz"
    for k, o in enumerate(OUTCOMES):
        ax = axes.flat[k]
        w = pm.pivot(index="ParticipantID", columns="PhaseID", values=o).reindex(columns=PHASES)
        for pid, r in w.iterrows(): ax.plot(range(4), r.values, color="#bbbbbb", lw=0.4, zorder=1)
        for p in PHASES:
            v = w[p].dropna().values
            ax.scatter(np.full(len(v), pos[p]), v, s=6, color=PHASE_COL[p], zorder=2, lw=0)
            ax.errorbar(pos[p] + 0.22, v.mean(), yerr=v.std(ddof=1) / np.sqrt(len(v)), fmt="s", ms=2.8, color="black", lw=0.7, capsize=1.2, zorder=3)
        for _, r in t[t.Outcome == o].iterrows():   # assert means agree with the exported contrast table
            assert abs(w[r.Reference_PhaseID].mean() - r.Reference_Mean_raw) <= 1e-6 * max(1, abs(r.Reference_Mean_raw))
        sig = t[(t.Outcome == o) & (t.FDR_significant.astype(str).str.upper() == "TRUE")]
        lo, hi = np.nanmin(w.values), np.nanmax(w.values); rng = hi - lo if hi > lo else 1; lev = 0
        for _, r in sig.sort_values("Contrast").iterrows():
            a_, b_ = sorted([pos[r.Reference_PhaseID], pos[r.Comparison_PhaseID]]); y = hi + rng * (0.08 + 0.13 * lev)
            ax.plot([a_, a_, b_, b_], [y - rng * 0.03, y, y, y - rng * 0.03], color="black", lw=0.6)
            ax.text((a_ + b_) / 2, y, "*" if r.q_FDR >= 0.01 else ("**" if r.q_FDR >= 0.001 else "***"), ha="center", va="bottom", fontsize=6.5, linespacing=0.6)
            lev += 1
        ax.set_ylim(lo - rng * 0.06, hi + rng * (0.12 + 0.13 * max(lev, 0.4)))
        ax.set_xticks(range(4)); ax.set_xticklabels(PHASES, fontsize=4.8, rotation=0)
        for tl, p in zip(ax.get_xticklabels(), PHASES): tl.set_color(PHASE_COL[p])
        ax.tick_params(axis="y", labelsize=5); ax.set_xlim(-0.45, 3.6)
        ax.set_title(ulab(o), fontsize=5.8, pad=2)
        ax.text(-0.33, 1.06, letters[k], transform=ax.transAxes, fontsize=8, fontweight="bold", va="bottom")
    for ax in axes.flat[26:]: ax.set_axis_off()
    axl = axes.flat[26]
    txt = ("Grey lines / coloured dots: participant × phase means\n(ln applied per 10 s epoch for ln outcomes).\n"
           "Black square: mean ± SE across participants.\n"
           "Brackets: primary H1.1 (BikeU–BikeG), H1.2 (WalkU–WalkG),\nH3.1 (BikeU–WalkU), H3.2 (BikeG–WalkG) contrasts with\n"
           "BH q < 0.05 (* q<0.05, ** q<0.01, *** q<0.001;\nfamily = 26 outcomes per contrast).\n"
           "Eye-tracker panels: 10 participants (no eye data for P13);\nWalkU 9 (P8 WalkU missing).")
    axl.text(0.0, 1.0, txt, transform=axl.transAxes, fontsize=5.6, va="top")
    save(fig, "FigureS2_outcomes_by_phase")

figS2()

## Figure S3 — Example 10 s traces (P10, all four phases)

Inputs (relative to `ROOT`):
* `output\merged_all_11participants_8min.csv`

In [ ]:
# Figure S3 — example 10 s traces (one participant, all four phases)
def figS3(pid="P10"):
    d = analysis_units(merged()); x = d[d.ParticipantID == pid]
    vars_ = [("ucm_UTCI", "UTCI (°C)"), ("ucm_SND_dBA", "Sound level (dBA)"), ("empatica_heart_rate", ulab("empatica_heart_rate")),
             ("empatica_eda_tonic", ulab("empatica_eda_tonic")), ("eyetracker_pupil_diameter_avg", ulab("eyetracker_pupil_diameter_avg"))]
    for v, _ in vars_: assert x[v].notna().mean() > 0.9, (v, x[v].notna().mean())
    fig, axes = plt.subplots(len(vars_), 4, figsize=(7.2, 7.4), sharey="row", sharex=True)
    fig.subplots_adjust(left=0.11, right=0.99, top=0.93, bottom=0.1, hspace=0.18, wspace=0.08)
    starts = {p: x[x.PhaseID == p].Datetime.min() for p in PHASES}
    for i, (v, lab) in enumerate(vars_):
        for j, p in enumerate(PHASES):
            ax = axes[i, j]; y = x[x.PhaseID == p]
            ax.plot(y.t_sec / 60, y[v].astype(float), color=PHASE_COL[p], lw=0.9)
            m = y[v].astype(float).mean(); ax.axhline(m, color=PHASE_COL[p], lw=0.6, ls="--")
            if i == 0: ax.set_title(f"{PHASE_LAB[p]} ({p})\nstart {starts[p].strftime('%H:%M')}", fontsize=6.5, color=PHASE_COL[p])
            if j == 0: ax.set_ylabel(lab, fontsize=6.3)
            if i == len(vars_) - 1: ax.set_xlabel("Minutes from phase start", fontsize=6.3)
            ax.tick_params(labelsize=5.6)
        axes[i, 0].text(-0.42, 1.0, "abcde"[i], transform=axes[i, 0].transAxes, fontsize=9, fontweight="bold", va="top")
    fig.text(0.11, 0.012, f"Participant {pid}, {x.Datetime.min().strftime('%d %b %Y')}; 48 × 10 s epochs per phase (clock time as stored in the merged file). Dashed line: phase mean.\n"
             "EDA tonic in natural-log analysis units, as in the models.", fontsize=5.6, color="#444444")
    save(fig, "FigureS3_example_traces")

figS3()

## Figure S4 — H5 corridor-cell (hotspot) results

Inputs (relative to `ROOT`):
* `output\statistical_analysis\results\H5_exploratory\H5_RH1_hotspot_FDR_all_tests.csv, H5_corridor_grid_label_map.csv, H5_epoch_buffer_assignment.csv`

In [ ]:
# Figure S4 — H5 corridor-cell (hotspot) results
def figS4():
    t = pd.read_csv(src("res/H5/H5_RH1_hotspot_FDR_all_tests.csv", r"output\statistical_analysis\results\H5_exploratory\H5_RH1_hotspot_FDR_all_tests.csv"))
    gm = pd.read_csv(src("res/H5/H5_corridor_grid_label_map.csv", r"output\statistical_analysis\results\H5_exploratory\H5_corridor_grid_label_map.csv"))
    ea = pd.read_csv(src("res/H5/H5_epoch_buffer_assignment.csv", r"output\statistical_analysis\results\H5_exploratory\H5_epoch_buffer_assignment.csv"))
    # verify the stored FDR = BH within outcome x phase
    for (o, p), g in t.groupby(["outcome", "PhaseID"]):
        assert np.allclose(bh(g.p_value.values), g.p_value_FDR.values, atol=1e-9, equal_nan=True)
    cen = ea.dropna(subset=["hotspot_id"]).groupby("hotspot_id")[["ucm_GPS_lat", "ucm_GPS_lon"]].median()
    fig = plt.figure(figsize=(7.2, 9.4))
    # a: count heatmap
    axa = fig.add_axes([0.23, 0.555, 0.30, 0.42])
    N = t.groupby(["outcome", "PhaseID"]).size().unstack().reindex(index=OUTCOMES, columns=PHASES)
    S = t[t.p_value_FDR < 0.05].groupby(["outcome", "PhaseID"]).size().unstack().reindex(index=OUTCOMES, columns=PHASES).fillna(0)
    im = axa.imshow((S / N * 100).values, cmap="Reds", vmin=0, vmax=100, aspect="auto")
    for i in range(26):
        for j in range(4):
            v = S.values[i, j] / N.values[i, j] * 100
            axa.text(j, i, f"{int(S.values[i, j])}/{int(N.values[i, j])}", ha="center", va="center", fontsize=4.6, color="white" if v > 60 else "black")
    axa.set_yticks(range(26)); axa.set_yticklabels([LAB[o] for o in OUTCOMES], fontsize=5.3)
    axa.set_xticks(range(4)); axa.set_xticklabels(PHASES, fontsize=6); axa.xaxis.tick_top(); axa.tick_params(length=0)
    for tl, p in zip(axa.get_xticklabels(), PHASES): tl.set_color(PHASE_COL[p])
    axa.axhline(12.5, color="black", lw=0.6)
    cb = fig.colorbar(im, cax=fig.add_axes([0.545, 0.555, 0.01, 0.42])); cb.set_label("% of cells with BH q < 0.05", fontsize=5.8); cb.ax.tick_params(labelsize=5)
    panel_label(axa, "a", dx=-0.16, dy=0.0)
    fig.text(0.6, 0.975, f"a: significant cell coefficients / tested cells\n(RH1, cell vs first corridor cell of the phase;\nBH q within outcome × phase; total {int(S.values.sum())} of {len(t)}).\n"
             f"Cells: corridor-buffer grid along each route\n({', '.join(f'{p} {int((gm.PhaseID == p).sum())}' for p in PHASES)}).\n"
             "H5 models use the outcomes on the raw scale\n(no ln transform in H5_hotspot_core.R).\n\n"
             "b–i: cell coefficient (difference from reference\ncell ★, raw units) at the median GPS position\nof the cell's 10 s epochs; black ring = q < 0.05.",
             fontsize=5.6, va="top")
    # maps
    lat0 = ea.ucm_GPS_lat.median(); kx = np.cos(np.deg2rad(lat0))
    rows = [("empatica_skin_temperature", "Skin temperature (°C)", "bcde"), ("empatica_eda_tonic", "EDA tonic (µS, raw scale)", "fghi")]
    for r, (o, lab, lets) in enumerate(rows):
        s = t[t.outcome == o]; vmax = np.nanpercentile(np.abs(s.estimate), 98)
        for j, p in enumerate(PHASES):
            ax = fig.add_axes([0.035 + j * 0.218, 0.295 - r * 0.25, 0.2, 0.205])
            g = gm[gm.PhaseID == p].merge(s[s.PhaseID == p][["grid_label", "estimate", "p_value_FDR"]], on="grid_label", how="left")
            g = g.join(cen, on="hotspot_id")
            ref = g[g.grid_number == 1]
            ax.plot(g.sort_values("route_bin").ucm_GPS_lon * kx, g.sort_values("route_bin").ucm_GPS_lat, color="#cccccc", lw=0.6, zorder=1)
            sc = ax.scatter(g.ucm_GPS_lon * kx, g.ucm_GPS_lat, c=g.estimate, cmap="RdBu_r", vmin=-vmax, vmax=vmax, s=14, zorder=2,
                            edgecolors=np.where(g.p_value_FDR < 0.05, "black", "none"), linewidths=0.7)
            ax.scatter(ref.ucm_GPS_lon * kx, ref.ucm_GPS_lat, marker="*", s=40, color="gold", edgecolors="black", linewidths=0.4, zorder=3)
            ax.set_aspect("equal", adjustable="datalim"); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_color("#999999"); sp.set_linewidth(0.5)
            ns = int((g.p_value_FDR < 0.05).sum()); nt = int(g.estimate.notna().sum())
            ax.set_title(f"{p}: {ns}/{nt} cells q<0.05", fontsize=6, color=PHASE_COL[p], pad=2)
            ax.text(0.0, 1.07, lets[j], transform=ax.transAxes, fontsize=9, fontweight="bold", ha="right", va="bottom")
        cax = fig.add_axes([0.905, 0.31 - r * 0.25, 0.01, 0.175]); cb = fig.colorbar(sc, cax=cax)
        cb.set_label(f"{lab}\ncell − reference cell", fontsize=5.6); cb.ax.tick_params(labelsize=5)
        fig.text(0.47, 0.532 - r * 0.25, lab + ": RH1 cell coefficients by phase", fontsize=6.8, ha="center", va="bottom", style="italic")
    fig.text(0.06, 0.008, f"{int(ea.hotspot_id.isna().sum())} of {len(ea)} epochs fall outside every corridor buffer and are not assigned to a cell. "
             "Map positions are cell medians of WGS84 GPS (longitude scaled by cos latitude); no basemap.", fontsize=5.4, color="#444444")
    save(fig, "FigureS4_H5_hotspots")

figS4()

## Figure S5 — H6 (exploratory, not preregistered) and H9 stress composites

Inputs (relative to `ROOT`):
* `output\statistical_analysis\results\H5_exploratory\H6_1_green_urban_contrasts.csv, H6_2_lag_profiles.csv, H6_3_participant_slopes.csv, H6_3_sound_source_adjusted.csv, H6_README.txt`
* `output\statistical_analysis\results\H6_stress_composite\H6_pairwise_rmcorr.csv, H6_composite_definitions.csv`
* `output\merged_all_11participants_8min.csv`

In [ ]:
# Figure S5 — H6 (exploratory, not preregistered) and H9 (stress composites)
OLAB = {"questionnaire_prs11_total_mean_1_5": "Restorativeness PRS-11 (1–5)", "eyetracker_fixation_rate": "Fixation rate (/s)",
        "eyetracker_fixation_duration_s": "Fixation duration (s)", "empatica_eda_tonic": "EDA tonic (µS, raw)",
        "eyetracker_pupil_diameter_avg": "pupil diameter", "empatica_skin_temperature": "skin temperature", "empatica_eda_phasic": "EDA phasic",
        "lys_lux_avg": "Δlux", "ucm_MRT": "ΔMRT", "ucm_SND_dBA": "ΔdBA", "vehicle_proportion": "Vehicle share", "rural_natural_proportion": "Rural/natural share"}

def figS5():
    base = r"output\statistical_analysis\results\H5_exploratory" + "\\"
    c1 = pd.read_csv(src("res/H6/H6_1_green_urban_contrasts.csv", base + "H6_1_green_urban_contrasts.csv"))
    l2 = pd.read_csv(src("res/H6/H6_2_lag_profiles.csv", base + "H6_2_lag_profiles.csv"))
    s3 = pd.read_csv(src("res/H6/H6_3_participant_slopes.csv", base + "H6_3_participant_slopes.csv"))
    a3 = pd.read_csv(src("res/H6/H6_3_sound_source_adjusted.csv", base + "H6_3_sound_source_adjusted.csv"))
    src("res/H6/H6_README.txt", base + "H6_README.txt")
    h9 = pd.read_csv(src("res/H6/H6_pairwise_rmcorr.csv", r"output\statistical_analysis\results\H6_stress_composite\H6_pairwise_rmcorr.csv"))
    defs = pd.read_csv(src("res/H6/H6_composite_definitions.csv", r"output\statistical_analysis\results\H6_stress_composite\H6_composite_definitions.csv"))
    # H6.1 estimates were re-derived from the current merged file (participant-phase means, raw scale) before plotting
    d = merged()
    for _, r in c1.iterrows():
        pm = d.groupby(["ParticipantID", "PhaseID"])[r.Outcome].mean().unstack()
        g, u = ("BikeG", "BikeU") if r.Mode == "Bike" else ("WalkG", "WalkU")
        x = (pm[g] - pm[u]).dropna(); assert len(x) == r.N and abs(x.mean() - r.estimate) < 1e-9
    fig = plt.figure(figsize=(7.2, 9.8))
    # a: H6.1 standardized paired differences
    ax = fig.add_axes([0.25, 0.80, 0.24, 0.175])
    order = list(dict.fromkeys(c1.Outcome))
    for k, o in enumerate(order):
        for mo, dy, col in [("Bike", -0.17, PHASE_COL["BikeG"]), ("Walk", 0.17, PHASE_COL["WalkG"])]:
            r = c1[(c1.Outcome == o) & (c1.Mode == mo)].iloc[0]; y = k + dy
            ax.plot([r.ci_low / r.sd_paired, r.ci_high / r.sd_paired], [y, y], color=col, lw=1)
            ax.plot(r.estimate / r.sd_paired, y, "o", ms=3.8, mfc=col if r.q < 0.05 else "white", mec=col)
            ax.text(5.6, y, f"{r.estimate:+.3g} (q={r.q:.3f})", fontsize=4.8, va="center")
    ax.axvline(0, color="#777777", ls="--", lw=0.6); ax.set_xlim(-2.2, 5.5)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([OLAB[o] for o in order], fontsize=5.8); ax.set_ylim(len(order) - 0.5, -0.5)
    ax.set_xlabel("Green − urban, d_z (paired mean / SD of differences)", fontsize=6)
    ax.legend(handles=[Line2D([], [], marker="o", color=PHASE_COL["BikeG"], label="Bike", ms=3.5), Line2D([], [], marker="o", color=PHASE_COL["WalkG"], label="Walk", ms=3.5)],
              frameon=False, fontsize=5.4, loc="lower right")
    ax.set_title("H6.1 green − urban (raw-unit estimate, q)", fontsize=6.4, loc="left")
    panel_label(ax, "a", dx=-0.19, dy=0.005)
    # b: H6.2 lag profiles
    ax = fig.add_axes([0.745, 0.80, 0.235, 0.175]); cols = ["#1f78b4", "#e31a1c", "#6a3d9a"]
    for (e, o), cc in zip(l2.groupby(["Exposure", "Outcome"], sort=False).groups, cols):
        g = l2[(l2.Exposure == e) & (l2.Outcome == o)].sort_values("Lag_s")
        ax.fill_between(g.Lag_s, g.ci_low, g.ci_high, color=cc, alpha=0.12, lw=0)
        ax.plot(g.Lag_s, g.mean_r, "-", color=cc, lw=1, label=f"{OLAB[e]} → Δ{OLAB[o]}")
        for _, r in g.iterrows(): ax.plot(r.Lag_s, r.mean_r, "o", ms=3.5, mec=cc, mfc=cc if r.q < 0.05 else "white")
    ax.axhline(0, color="#777777", ls="--", lw=0.6); ax.set_xticks([0, 10, 30, 60])
    ax.set_xlabel("Lag (s)", fontsize=6); ax.set_ylabel("Mean within-person r (95% CI)", fontsize=6)
    ax.legend(frameon=False, fontsize=4.8, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=1)
    ax.set_title("H6.2 first-difference lag profiles", fontsize=6.4, loc="left")
    panel_label(ax, "b", dx=-0.06, dy=0.005)
    # c: H6.3 participant slopes
    ax = fig.add_axes([0.25, 0.575, 0.24, 0.15])
    for k, (_, r) in enumerate(a3.iterrows()):
        s = s3[(s3.Source == r.Source) & (s3.Outcome == r.Outcome)]
        ax.scatter(s.Beta_SD, np.full(len(s), k) + np.linspace(-0.15, 0.15, len(s)), s=5, color="#888888", lw=0)
        ax.plot([r.ci_low, r.ci_high], [k, k], color="black", lw=1); ax.plot(r.mean_beta_SD, k, "D", ms=3.5, mfc="black" if r.q < 0.05 else "white", mec="black")
        ax.text(1.02, k, f"q={r.q:.2f}", transform=ax.get_yaxis_transform(), fontsize=5, va="center")
    ax.axvline(0, color="#777777", ls="--", lw=0.6)
    ax.set_yticks(range(len(a3))); ax.set_yticklabels([f"{OLAB[r.Source]} → {OLAB[r.Outcome]}" for _, r in a3.iterrows()], fontsize=5.6); ax.set_ylim(len(a3) - 0.5, -0.5)
    ax.set_xlabel("Within-bout standardized slope (per participant; mean, 95% CI)", fontsize=6)
    ax.set_title("H6.3 sound source adjusted for dBA, UTCI, lux", fontsize=6.4, loc="left")
    panel_label(ax, "c", dx=-0.19, dy=0.005)
    fig.text(0.6, 0.72, "H6 is exploratory and NOT preregistered (H6_README).\nH6 README records input MD5 13ffdfbd… (D: path), not\nthe current d1566f… file; the H6.1 estimates in a were\nre-derived from the current file and match exactly.\nH6 uses outcomes on the raw scale (no ln transform).\nFilled symbols: BH q < 0.05 within each H6 family.",
             fontsize=5.6, va="top", color="#333333")
    # d: H9 rmcorr — stress-oriented physiological/ocular composites × environmental composites
    triv = (h9.r_rm.abs() > 0.999999)
    assert triv.sum() == 4
    rows_ = list(defs[defs.domain.isin(["physiological", "ocular_transparent", "ocular_proprietary"])].composite_id)
    cols_ = list(defs[defs.domain.isin(["environmental", "environmental_group"])].composite_id)
    R = np.full((len(rows_), len(cols_)), np.nan); Q = np.full_like(R, np.nan); NF = np.zeros(R.shape, bool)
    key = {}
    for _, r in h9.iterrows(): key[(r.Composite_X, r.Composite_Y)] = r; key[(r.Composite_Y, r.Composite_X)] = r
    for i, a_ in enumerate(rows_):
        for j, b_ in enumerate(cols_):
            r = key[(a_, b_)]
            if r.Status != "FIT": NF[i, j] = True; continue
            R[i, j] = r.r_rm; Q[i, j] = r.q_FDR
    ax = fig.add_axes([0.25, 0.1, 0.66, 0.395])
    im = ax.imshow(R, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto", interpolation="nearest")
    yy, xx = np.where(Q < 0.05); ax.scatter(xx, yy, s=2.2, color="black", lw=0)
    yy, xx = np.where(NF); ax.scatter(xx, yy, marker="x", s=6, color="#888888", lw=0.5)
    lab = lambda c: (ulab(c.split("__", 1)[1]) if c.startswith("physio__") else
                     {"eye__transparent_stress_index": "Ocular stress index (4 metrics)", "eye__proprietary_stress_composite": "Eye-tracker stress composite (vendor)"}.get(c, c.replace("env__", "").replace("env_group_", "group: ")))
    ax.set_yticks(range(len(rows_))); ax.set_yticklabels([lab(c) for c in rows_], fontsize=4.8)
    ax.set_xticks(range(len(cols_))); ax.set_xticklabels([lab(c) for c in cols_], rotation=60, ha="right", rotation_mode="anchor", fontsize=4.8)
    ax.axhline(25.5, color="black", lw=0.5); ax.axvline(20.5, color="black", lw=0.5); ax.tick_params(length=0)
    cb = fig.colorbar(im, cax=fig.add_axes([0.925, 0.1, 0.01, 0.395])); cb.set_label("rmcorr r (stress-oriented, within-person z)", fontsize=5.8); cb.ax.tick_params(labelsize=5)
    nfit = int((h9.Status == "FIT").sum()); nsig = int((h9.FDR_significant == True).sum())
    ax.set_title(f"H9 composite rmcorr, physiological/ocular × environmental block ({int(np.isfinite(R).sum())} fitted pairs; {int((Q < 0.05).sum())} with q < 0.05)", fontsize=6.4, loc="left")
    panel_label(ax, "d", dx=-0.19, dy=0.01)
    fig.text(0.02, 0.004, f"d: dot = BH q < 0.05; × = not fitted (insufficient data: env atmo_pm1_avg and lys_medi_avg composites are empty).\nBH family = all {nfit} fitted pairs of the 55 composites ({nsig} q < 0.05), which includes 4 trivial r = 1 pairs (LF nu vs HF nu; dBA, PM2.5 and speed vs their\n"
             "single-member groups) that are not shown. H9 applies ln to participant × phase means (H1–H4 average ln over 10 s epochs).",
             fontsize=5.2, color="#444444")
    save(fig, "FigureS5_H6_H9")

figS5()

## Figure S6 — Exposure-screening PCA diagnostics

Inputs (relative to `ROOT`):
* `output\statistical_analysis\Normalization & Parameter Screening\PCA-informed\All-columns two PCAs\{air_temp,sound}_pca_{scree,biplot,loadings}.png`

In [ ]:
# Figure S6 — exposure-screening diagnostics (existing PCA outputs, composited)
def figS6():
    pdir = r"output\statistical_analysis\Normalization & Parameter Screening\PCA-informed\All-columns two PCAs" + "\\"
    fig = plt.figure(figsize=(7.2, 9.5))
    hs = [0.215, 0.25, 0.43]; gap = 0.03; y = 0.985; letters = iter("abcdef")
    for kind, h in zip(["scree", "biplot", "loadings"], hs):
        y -= h
        for j, grp in enumerate(["air_temp", "sound"]):
            ax = fig.add_axes([0.03 + j * 0.49, y, 0.46, h - 0.01])
            img_axes(ax, src(f"art/pca/{grp}_pca_{kind}.png", pdir + f"{grp}_pca_{kind}.png"))
            fig.text(0.03 + j * 0.49, y + h - 0.01, next(letters), fontsize=10, fontweight="bold", ha="right", va="top")
        y -= gap * (0.3 if kind == "scree" else 0.5)
    fig.text(0.03, 0.005, "Left: air-temperature PCA (air_temp: ucm_AIR_temp and the three AIR_T heights); right: sound PCA (22 sound measures + dBA).\nRows: scree, correlation biplot, standardized PC1/PC2 weights. Images reproduced unchanged from the screening outputs.",
             fontsize=5.6, color="#444444")
    save(fig, "FigureS6_PCA_diagnostics")

figS6()

## Input files read in this session

In [ ]:
for p in sorted(USED): print(p)